# FINA4030A — Lab 1
## The delegation problem

**Class 1.** Not graded. Keep it — you will refer back to it in Class 9.

There is no API key, no secret and no installation in this lab. Everything happens in a chat window you already have, plus this notebook to record what you saw. That is deliberate: on day one nothing should depend on infrastructure.

**The question.** Most people ask of an AI system: *can it do the task?* That is the wrong question, and this lab is designed to show you why.

The right question is: **how much authority should I grant it, and what do I give up at each level?** You are about to run one task three times, granting more authority each time, and measure what happens to your ability to see whether the work is any good.

**What to expect.** Almost everyone finds the same thing, and it is uncomfortable: the level that produces the most confident, most polished, fastest answer is also the level at which you have the least idea whether it is right.

---

## The task

You are producing a **peer comparables screen**: identify five listed peers for a target company and report the median EV/EBITDA multiple, with a one-line justification for the peer set.

This is a real analyst task, it is small enough to do three times in forty minutes, and — importantly — it has no single correct answer. Reasonable analysts disagree about peer sets. That matters, because it means you cannot fall back on "the right answer" and must judge the *work* instead.

Your target company is on the board. Write it in below.

In [ ]:
TARGET = ""        # your instructor will give you this at the start of the lab

print(f"Target company: {TARGET or '[not filled in]'}")
print()
print("The prompt you will use, at all three levels:")
print("-" * 60)
TASK = f"""Identify five listed peer companies for {TARGET}.
For each peer, give the company name, ticker, and current EV/EBITDA.
Then report the median EV/EBITDA of the five.
Finish with one sentence justifying the peer set you chose."""
print(TASK)

---

## Level 1 — Read-only assistance

**You do the work. The model answers questions.**

You may ask it anything — how peer sets are normally constructed for this sector, what EV/EBITDA is sensitive to, whether a particular company belongs. But **you** decide the peer set, and **you** write down the answer.

Do not paste the task above. Ask questions instead.

Give yourself **12 minutes**. Note the time you start.

---

## Level 2 — Propose and approve

**The model proposes. You check every step before accepting it.**

Now paste the task. But before you accept anything, work through the reply line by line:

- Is each company actually a peer? Would you defend that choice to a portfolio manager?
- Where did each multiple come from? Ask it.
- Is the median arithmetically correct? Compute it yourself.
- Does the justification actually justify anything, or is it a sentence shaped like a justification?

Push back where you disagree, and note each time you do.

Give yourself **12 minutes**.

---

## Level 3 — Bounded autonomous execution

**The model does the whole thing. You see only the result.**

Paste the task in a **fresh conversation**. Read the answer once, as a busy person would. Do not interrogate it, do not check the arithmetic, do not ask follow-ups.

Then write it down and move on.

Give yourself **4 minutes** — and notice how much of that you did not need.

---

## Record what happened

Fill this in honestly. Nobody marks it and there is no good answer to have. Guessing at what you think you should have found defeats the purpose.

In [ ]:
RUNS = {
    "level_1_readonly": {
        "minutes":            None,   # how long it actually took
        "median_ev_ebitda":   None,   # the number you ended up with
        "peers":              [],     # e.g. ["AAA", "BBB", ...]
        "steps_you_saw":      None,   # how many distinct steps were visible to you
        "errors_you_caught":  None,   # things you spotted and fixed or rejected
        "your_confidence":    None,   # 1 = would not sign it, 5 = would put my name on it
    },
    "level_2_approve": {
        "minutes":            None,
        "median_ev_ebitda":   None,
        "peers":              [],
        "steps_you_saw":      None,
        "errors_you_caught":  None,
        "your_confidence":    None,
    },
    "level_3_autonomous": {
        "minutes":            None,
        "median_ev_ebitda":   None,
        "peers":              [],
        "steps_you_saw":      None,
        "errors_you_caught":  None,
        "your_confidence":    None,
    },
}

missing = [f"{k}.{f}" for k, v in RUNS.items() for f, x in v.items()
           if x is None or x == []]
print(f"{len(missing)} field(s) still to fill in" if missing else "all fields filled")
for m in missing[:6]:
    print("   ", m)

### What the three runs tell you

In [ ]:
import pandas as pd

df = pd.DataFrame(RUNS).T
df.index = ["1 read-only", "2 propose+approve", "3 autonomous"]
display(df[["minutes", "median_ev_ebitda", "steps_you_saw",
            "errors_you_caught", "your_confidence"]])

med = [v["median_ev_ebitda"] for v in RUNS.values() if v["median_ev_ebitda"]]
if len(med) == 3:
    print(f"\nYour three medians: {', '.join(f'{m:.1f}' for m in med)}")
    print(f"range: {max(med) - min(med):.1f}  "
          f"({(max(med) - min(med)) / (sum(med)/3) * 100:.0f}% of the mean)")
    if max(med) - min(med) > 0.01:
        print("\nThree runs of one task, three different answers — and you have")
        print("no external source telling you which, if any, is right.")

# how much overlap between the peer sets?
sets = [set(v["peers"]) for v in RUNS.values() if v["peers"]]
if len(sets) == 3:
    common = set.intersection(*sets)
    everyone = set.union(*sets)
    print(f"\npeers named at least once : {len(everyone)}")
    print(f"peers common to all three  : {len(common)}  {sorted(common)}")
    if len(common) < 3:
        print("\nThe peer sets barely agree with each other. The median is a")
        print("statistic computed over a set that was itself unstable.")

### The shape of the trade

In [ ]:
import matplotlib.pyplot as plt

lvl = [1, 2, 3]
mins   = [RUNS[k]["minutes"] for k in RUNS]
seen   = [RUNS[k]["steps_you_saw"] for k in RUNS]
caught = [RUNS[k]["errors_you_caught"] for k in RUNS]
conf   = [RUNS[k]["your_confidence"] for k in RUNS]

if all(x is not None for x in mins + seen + caught + conf):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    ax[0].plot(lvl, mins, "o-", color="#1f3b73", label="minutes spent")
    ax[0].plot(lvl, seen, "s-", color="#b03a2e", label="steps you could see")
    ax[0].plot(lvl, caught, "^-", color="#6b7a3a", label="errors you caught")
    ax[0].set_xticks(lvl); ax[0].set_xlabel("autonomy granted")
    ax[0].legend(fontsize=8); ax[0].grid(alpha=.25)
    ax[0].set_title("What you spend, and what you see", fontsize=10)

    ax[1].bar([str(l) for l in lvl], conf, color="#1f3b73", alpha=.85)
    ax[1].set_ylim(0, 5.5); ax[1].set_xlabel("autonomy granted")
    ax[1].set_ylabel("your confidence (1–5)")
    ax[1].set_title("How sure you felt", fontsize=10); ax[1].grid(alpha=.25, axis="y")
    fig.tight_layout(); plt.show()

    print("If your confidence did not fall as fast as your visibility did,")
    print("that gap is the thing this course is about.")
else:
    print("Fill in RUNS above, then run this cell.")

---

## Naming what you just varied

You changed one thing between the three runs — how much authority you granted. But "authority" is not one dial. It is at least seven, and we will use these all term.

For each of your three runs, work out where it sat on each axis. Some will be identical across all three; the interesting ones are those that moved.

| Axis | What it asks |
|---|---|
| **Execution authority** | Can it act, or only propose? |
| **Reversibility and blast radius** | If it is wrong, what breaks, and can you undo it? |
| **Horizon at a reliability threshold** | How long a task can it complete *reliably enough*, not just once? |
| **Verification regime** | What checks run, and who runs them? |
| **Tool and data surface** | What can it reach? |
| **Oversight mode** | Are you in the loop, on the loop, or out of it? |
| **Determinism guarantee** | Would you get the same answer twice? |

The last one you cannot answer yet. You will measure it in Class 2, and the answer will surprise you.

In [ ]:
AXES = {
    "execution_authority":   {"level_1": "", "level_2": "", "level_3": ""},
    "reversibility":         {"level_1": "", "level_2": "", "level_3": ""},
    "horizon":               {"level_1": "", "level_2": "", "level_3": ""},
    "verification_regime":   {"level_1": "", "level_2": "", "level_3": ""},
    "tool_and_data_surface": {"level_1": "", "level_2": "", "level_3": ""},
    "oversight_mode":        {"level_1": "", "level_2": "", "level_3": ""},
    "determinism":           {"level_1": "unknown", "level_2": "unknown",
                              "level_3": "unknown"},
}

import pandas as pd
display(pd.DataFrame(AXES).T)
print("Which axes actually moved between your runs? Which stayed put?")
print("The ones that stayed put are the ones you did not think to vary.")

---

## Before you leave

**One sentence, written now, while it is fresh.** Which level would you use for real work, and what would have to be true for you to trust the output?

```
YOUR ANSWER:
```

Keep this notebook. In Class 9 you will write a governance document that has to specify, for a system your team built, exactly what you specified informally here. It will be easier if you can see what you thought on day one.

---

### What happens next

**Class 2** takes the last axis — determinism — and measures it. You will send one completely specified valuation ten times at the setting that is supposed to guarantee an identical answer, and count how many different answers come back.

Before then you need your CUHK API Portal key working, which we set up in the last few minutes of today's class.
